## Data collection
Some data's were collected manually through websites and some through API endpoints. Data collected through API's can be reproduced through the scripts below. These are cleaned scripts, data were collected and verified through messier scripts.

In [1]:
import json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd

RAW = Path("raw/fifa"); RAW.mkdir(parents=True, exist_ok=True)
MANIFEST = Path("raw/manifest.csv")
HEADERS = {"User-Agent": "Mozilla/5.0"}
COMP, SEASON = 17, 285023

def fetch_json(url, params, out):
    # Reuse the saved file if it exists, so a re-run can never pull different data.
    if out.exists():
        return json.loads(out.read_text(encoding="utf-8"))
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    out.write_text(r.text, encoding="utf-8")
    row = pd.DataFrame([{"file": str(out), "url": r.url,
                         "fetched_utc": datetime.now(timezone.utc).isoformat(),
                         "sha256": hashlib.sha256(r.content).hexdigest()}])
    row.to_csv(MANIFEST, mode="a", header=not MANIFEST.exists(), index=False)
    return r.json()

cal = fetch_json("https://api.fifa.com/api/v3/calendar/matches",
                 {"idCompetition": COMP, "idSeason": SEASON, "count": 500, "language": "en"},
                 RAW / "calendar.json")
matches = cal["Results"]

print(len(matches))
print(pd.Series([m["ResultType"] for m in matches]).value_counts())
print(sorted(matches[0].keys()))

104
1    95
3     5
2     4
Name: count, dtype: int64
['AggregateAwayTeamScore', 'AggregateHomeTeamScore', 'Attendance', 'Away', 'AwayTeamPenaltyScore', 'AwayTeamScore', 'BallPossession', 'CompetitionName', 'Date', 'FirstHalfExtraTime', 'FirstHalfTime', 'GroupName', 'Home', 'HomeTeamPenaltyScore', 'HomeTeamScore', 'IdCompetition', 'IdGroup', 'IdMatch', 'IdSeason', 'IdStage', 'IsHomeMatch', 'IsTicketSalesAllowed', 'IsUpdateable', 'LastPeriodUpdate', 'Leg', 'LocalDate', 'MatchDay', 'MatchLegInfo', 'MatchNumber', 'MatchReportUrl', 'MatchStatus', 'MatchTime', 'OfficialityStatus', 'Officials', 'PlaceHolderA', 'PlaceHolderB', 'Properties', 'ResultType', 'SeasonName', 'SeasonShortName', 'SecondHalfExtraTime', 'SecondHalfTime', 'Stadium', 'StageName', 'TimeDefined', 'Weather', 'Winner']


In [2]:
print(sorted(matches[0]["Home"].keys()))
print(sorted(matches[0]["Stadium"].keys()))
print(sorted({m["StageName"][0]["Description"] for m in matches}))
print(sorted(m["MatchNumber"] for m in matches) == list(range(1, 105)))

['Abbreviation', 'AgeType', 'FootballType', 'Gender', 'IdAssociation', 'IdCountry', 'IdTeam', 'PictureUrl', 'Score', 'ShortClubName', 'Side', 'Tactics', 'TeamName', 'TeamType']
['AffiliationCountry', 'AffiliationRegion', 'Built', 'Capacity', 'CityName', 'Email', 'Fax', 'IdCity', 'IdCountry', 'IdStadium', 'IsUpdateable', 'Latitude', 'Length', 'Longitude', 'Name', 'Phone', 'PostalCode', 'Properties', 'Roof', 'Street', 'Turf', 'WebAddress', 'Width']
['Bronze final', 'Final', 'First Stage', 'Quarter-final', 'Round of 16', 'Round of 32', 'Semi-final']
True


In [3]:
STAGE_MAP = {"First Stage": "Group", "Round of 32": "Round of 32", "Round of 16": "Round of 16",
             "Quarter-final": "QF", "Semi-final": "SF", "Bronze final": "3rd place", "Final": "Final"}
STAGE_ORD = {"Group": 0, "Round of 32": 1, "Round of 16": 2, "QF": 3, "SF": 4, "Final": 5, "3rd place": 5}

rows = []
for m in matches:
    for side, opp in (("Home", "Away"), ("Away", "Home")):
        rows.append({
            "match_no": m["MatchNumber"],
            "fifa_match_id": int(m["IdMatch"]),
            "kickoff_utc": pd.Timestamp(m["Date"]),
            "stage": STAGE_MAP[m["StageName"][0]["Description"]],
            "team": m[side]["TeamName"][0]["Description"],
            "team_code": m[side]["IdCountry"],
            "opponent": m[opp]["TeamName"][0]["Description"],
            "opp_code": m[opp]["IdCountry"],
            "goals": m[f"{side}TeamScore"],
            "result_type": m["ResultType"],
            "venue": m["Stadium"]["Name"][0]["Description"],
            "city": m["Stadium"]["CityName"][0]["Description"],
            "venue_country": m["Stadium"]["IdCountry"],
        })

df = pd.DataFrame(rows).sort_values(["match_no", "team"]).reset_index(drop=True)
df["team_match_no"] = df.groupby("team_code")["kickoff_utc"].rank(method="first").astype(int)
df["went_to_et"] = df["result_type"] != 1
df["knockout"] = (df["stage"] != "Group").astype(int)
df["stage_ord"] = df["stage"].map(STAGE_ORD)

In [4]:
import time

live = {}
for m in matches:
    if m["ResultType"] != 1:
        url = f"https://api.fifa.com/api/v3/live/football/{COMP}/{SEASON}/{m['IdStage']}/{m['IdMatch']}"
        live[m["IdMatch"]] = fetch_json(url, {"language": "en"}, RAW / f"live_{m['IdMatch']}.json")
        time.sleep(0.2)

def find_key(obj, key, path=""):
    # Walks the nested JSON and prints every path where `key` appears.
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{path}.{k}"
            if k == key:
                print(p, type(v).__name__, len(v) if isinstance(v, list) else "")
            find_key(v, key, p)
    elif isinstance(obj, list) and obj:
        find_key(obj[0], key, path + "[0]")

print(len(live))
lm = next(iter(live.values()))
find_key(lm, "Goals")

9
.HomeTeam.Goals list 4
.AwayTeam.Goals list 5


 Knockout matches level after 90 minutes go to 30 minutes of extra time,
giving those teams more time to score than any group-stage team. Counting extra-time goals would inflate knockout
rows for a reason no pre-match feature can capture, since whether extra time happens depends on the match result
itself. We therefore count only regulation-time goals (first and second half, including stoppage time) so every row
covers the same playing time. Penalty shoot-out kicks are excluded because they are not goals in the match score

In [5]:
def goal_table(feed):
    # Pool both lists and credit by IdTeam rather than by list, so own goals land with the team that benefits.
    gs = feed["HomeTeam"]["Goals"] + feed["AwayTeam"]["Goals"]
    return pd.DataFrame([{"team_id": g["IdTeam"], "period": g["Period"]} for g in gs],
                        columns=["team_id", "period"])

g90 = {}
for m in matches:
    if m["ResultType"] == 1:
        continue
    gl = goal_table(live[m["IdMatch"]])
    assert set(gl["period"]) <= {3, 5, 7, 9, 11}, (m["IdMatch"], set(gl["period"]))
    for side in ("Home", "Away"):
        mine = gl[gl["team_id"] == m[side]["IdTeam"]]
        assert mine["period"].isin([3, 5, 7, 9]).sum() == m[f"{side}TeamScore"], (m["IdMatch"], side)
        assert (mine["period"] == 11).sum() == (m[f"{side}TeamPenaltyScore"] or 0), (m["IdMatch"], side)
        g90[(int(m["IdMatch"]), m[side]["IdCountry"])] = int(mine["period"].isin([3, 5]).sum())

assert len(g90) == 18
df["goals_90"] = df.apply(lambda r: r["goals"] if r["result_type"] == 1
                          else g90[(r["fifa_match_id"], r["team_code"])], axis=1)

print((df["goals_90"] != df["goals"]).sum(), "rows where extra-time goals were removed")
COLS = ["match_no", "fifa_match_id", "kickoff_utc", "stage", "team", "team_code", "opponent", "opp_code",
        "team_match_no", "goals", "goals_90", "went_to_et", "knockout", "stage_ord",
        "venue", "city", "venue_country"]
df[COLS].to_csv("obj2.2.csv", index=False)

6 rows where extra-time goals were removed


In [6]:
import pandas as pd
df = pd.read_csv("obj2.2.csv")
df.columns.to_list()

['match_no',
 'fifa_match_id',
 'kickoff_utc',
 'stage',
 'team',
 'team_code',
 'opponent',
 'opp_code',
 'team_match_no',
 'goals',
 'goals_90',
 'went_to_et',
 'knockout',
 'stage_ord',
 'venue',
 'city',
 'venue_country']

In [7]:
df.head()

,match_no,fifa_match_id,kickoff_utc,stage,team,team_code,opponent,opp_code,team_match_no,goals,goals_90,went_to_et,knockout,stage_ord,venue,city,venue_country
0,1,400021443,2026-06-11 19:00:00+00:00,Group,Mexico,MEX,South Africa,RSA,1,2,2,False,0,0,Mexico City Stadium,Mexico City,MEX
1,1,400021443,2026-06-11 19:00:00+00:00,Group,South Africa,RSA,Mexico,MEX,1,0,0,False,0,0,Mexico City Stadium,Mexico City,MEX
2,2,400021441,2026-06-12 02:00:00+00:00,Group,Czechia,CZE,Korea Republic,KOR,1,1,1,False,0,0,Guadalajara Stadium,Guadalajara,MEX
3,2,400021441,2026-06-12 02:00:00+00:00,Group,Korea Republic,KOR,Czechia,CZE,1,2,2,False,0,0,Guadalajara Stadium,Guadalajara,MEX
4,3,400021449,2026-06-12 19:00:00+00:00,Group,Bosnia and Herzegovina,BIH,Canada,CAN,1,1,1,False,0,0,Toronto Stadium,Toronto,CAN


In [8]:
df.shape

(208, 17)

In [9]:
# FIFA ranking released before WC2026 (published 11 June, 2026)
rk = fetch_json("https://inside.fifa.com/api/ranking-overview",
                {"locale": "en", "dateId": "id15136", "rankingType": "football"},
                RAW / "ranking_id15136.json")
dates = {r["lastUpdateDate"] for r in rk["rankings"]}
released = pd.Timestamp(dates.pop())
first_kickoff = pd.to_datetime(df["kickoff_utc"]).min()
assert first_kickoff > released  #Pre WC

pts = pd.DataFrame([{"code": r["rankingItem"]["countryCode"],
                     "points": r["rankingItem"]["totalPoints"]} for r in rk["rankings"]])
missing = set(df["team_code"]) - set(pts["code"])
assert not missing, missing

p = pts.set_index("code")["points"]
new = df[["match_no", "team_code"]].assign(fifa_points_own=df["team_code"].map(p),
                                           fifa_points_opp=df["opp_code"].map(p))

In [10]:
# Helper func to append
def append_cols(new, path="obj2.2.csv"):
    # Adds columns keyed on (match_no, team_code); refuses if existing data would change or columns already exist.
    base = pd.read_csv(path)
    added = [c for c in new.columns if c not in ("match_no", "team_code")]
    clash = set(added) & set(base.columns)
    assert not clash, f"already in file: {clash} — re-run from the Block 1 save cell"
    out = base.merge(new, on=["match_no", "team_code"], how="left", validate="one_to_one")
    assert len(out) == len(base) and out[base.columns].equals(base)
    assert out[added].notna().all().all(), "missing values in new columns"
    out.to_csv(path, index=False)
    return out
out = append_cols(new)


In [11]:
print(out.columns.tolist())
print(pd.read_csv("obj2.2.csv").columns.tolist())

['match_no', 'fifa_match_id', 'kickoff_utc', 'stage', 'team', 'team_code', 'opponent', 'opp_code', 'team_match_no', 'goals', 'goals_90', 'went_to_et', 'knockout', 'stage_ord', 'venue', 'city', 'venue_country', 'fifa_points_own', 'fifa_points_opp']
['match_no', 'fifa_match_id', 'kickoff_utc', 'stage', 'team', 'team_code', 'opponent', 'opp_code', 'team_match_no', 'goals', 'goals_90', 'went_to_et', 'knockout', 'stage_ord', 'venue', 'city', 'venue_country', 'fifa_points_own', 'fifa_points_opp']


In [12]:
# Elo ratings per match
def fetch_raw(url, params, out, headers=HEADERS):
    if out.exists():
        return out.read_text(encoding="utf-8")
    r = requests.get(url, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    r.encoding = "utf-8"
    out.write_text(r.text, encoding="utf-8")
    row = pd.DataFrame([{"file": str(out), "url": r.url,
                         "fetched_utc": datetime.now(timezone.utc).isoformat(),
                         "sha256": hashlib.sha256(r.content).hexdigest()}])
    row.to_csv(MANIFEST, mode="a", header=not MANIFEST.exists(), index=False)
    return r.text

def fetch_json(url, params, out, headers=HEADERS):
    return json.loads(fetch_raw(url, params, out, headers))

In [13]:
from io import StringIO
ELO = Path("raw/elo"); ELO.mkdir(parents=True, exist_ok=True)

teams_txt = fetch_raw("https://www.eloratings.net/en.teams.tsv", None, ELO / "en.teams.tsv")
arg_txt = fetch_raw("https://www.eloratings.net/Argentina.tsv", None, ELO / "Argentina.tsv")

print(teams_txt[:500])
print(pd.read_csv(StringIO(arg_txt), sep="\t", header=None).tail(8))

DN	Aden
AF	Afghanistan
AL	Albania
DZ	Algeria
AD	Andorra
AO	Angola
AI	Anguilla
AG	Antigua and Barbuda	Antigua & Barbuda	Antigua/Barbuda	Antigua/Barb
AR	Argentina
AM	Armenia
AW	Aruba
AU	Australia
AT	Austria
AH	Austria-Hungary
AZ	Azerbaijan
BS	Bahamas
BS_loc	in the Bahamas	in Bahamas
BH	Bahrain
BD	Bangladesh
BB	Barbados
UT	Basutoland
BY	Belarus
BC	Belgian Congo	Belg Congo
BE	Belgium
BZ	Belize
OR	Belorussia
BJ	Benin
BM	Bermuda
BT	Bhutan
HA	Bohemia
EM	Bohemia and Moravia	Bohemia & Moravia	Bohemia/Mor
        0   1   2   3   4   5   6   7    8   9     10    11  12  13  14  15
1114  2026   7   3  AR  CV   3   2  WC   US   3  2151  1619   0  −1   2  59
1115  2026   7   7  AR  EG   3   2  WC   US   5  2156  1742   0  −3   2  34
1116  2026   7  11  AR  CH   3   1  WC   US  21  2177  1928  +1  −1   2  11
1117  2026   7  15  AR  EN   2   1  WC   US  23  2200  2074   0   0   2   4
1118  2026   7  19  ES  AR   1   0  WC   US  27  2259  2173   0   0   1   2
1119  2026   9  30  AR  BO   4   0   F  NaN

In [14]:
alias, primary = {}, {}
for line in teams_txt.splitlines():
    code, *names = line.split("\t")
    if code.endswith("_loc"):
        continue
    primary[code] = names[0]
    for n in names:
        alias[n] = code

fifa_teams = sorted(df["team"].unique())
unmatched = [t for t in fifa_teams if t not in alias]
print(len(fifa_teams), unmatched)

48 ['Cabo Verde', 'Congo DR', "Côte d'Ivoire", 'IR Iran', 'Korea Republic', 'Türkiye']


In [15]:
RENAME = {"Cabo Verde": "Cape Verde", "Congo DR": "DR Congo", "Côte d'Ivoire": "Ivory Coast",
          "IR Iran": "Iran", "Korea Republic": "South Korea", "Türkiye": "Turkey"}

bad = [f for f, e in RENAME.items() if e not in alias]
assert not bad, bad

elo_code = {t: alias[RENAME.get(t, t)] for t in fifa_teams}
assert len(set(elo_code.values())) == 48
print(elo_code)

{'Algeria': 'DZ', 'Argentina': 'AR', 'Australia': 'AU', 'Austria': 'AT', 'Belgium': 'BE', 'Bosnia and Herzegovina': 'BA', 'Brazil': 'BR', 'Cabo Verde': 'CV', 'Canada': 'CA', 'Colombia': 'CO', 'Congo DR': 'CD', 'Croatia': 'HR', 'Curaçao': 'CW', 'Czechia': 'CZ', "Côte d'Ivoire": 'CI', 'Ecuador': 'EC', 'Egypt': 'EG', 'England': 'EN', 'France': 'FR', 'Germany': 'DE', 'Ghana': 'GH', 'Haiti': 'HT', 'IR Iran': 'IR', 'Iraq': 'IQ', 'Japan': 'JP', 'Jordan': 'JO', 'Korea Republic': 'KR', 'Mexico': 'MX', 'Morocco': 'MA', 'Netherlands': 'NL', 'New Zealand': 'NZ', 'Norway': 'NO', 'Panama': 'PA', 'Paraguay': 'PY', 'Portugal': 'PT', 'Qatar': 'QA', 'Saudi Arabia': 'SA', 'Scotland': 'SQ', 'Senegal': 'SN', 'South Africa': 'ZA', 'Spain': 'ES', 'Sweden': 'SE', 'Switzerland': 'CH', 'Tunisia': 'TN', 'Türkiye': 'TR', 'USA': 'US', 'Uruguay': 'UY', 'Uzbekistan': 'UZ'}


In [16]:
# Fetch elo history
FILE_OVERRIDE = {"CW": "Curacao"}  # eloratings file names drop accents (Curaçao → 404)

failed = {}
elo_txt = {}
for team, code in elo_code.items():
    fname = FILE_OVERRIDE.get(code, primary[code]).replace(" ", "_")
    try:
        elo_txt[code] = fetch_raw(f"https://www.eloratings.net/{fname}.tsv", None, ELO / f"{fname}.tsv")
    except requests.HTTPError as e:
        failed[team] = (fname, e.response.status_code)
    time.sleep(0.2)

print(len(elo_txt), failed)

48 {}


In [17]:
# Extract each team's WC2026 rows, convert post-match ratings to pre-match, and attach them to the match rows. Pre-match Elo is asserted to equal each team's rating after its previous match (no information from the match itself)
def num(s):
    # eloratings writes negatives with a Unicode minus (−), which pandas reads as text
    return pd.to_numeric(s.astype(str).str.replace("−", "-"))

parts = []
for code, txt in elo_txt.items():
    t = pd.read_csv(StringIO(txt), sep="\t", header=None)
    chg, post_h, post_a = num(t[9]), num(t[10]), num(t[11])
    home = t[3] == code
    t["opp"] = t[4].where(home, t[3])
    t["elo_own"] = (post_h - chg).where(home, post_a + chg)
    t["elo_opp"] = (post_a + chg).where(home, post_h - chg)
    t["prev_post"] = post_h.where(home, post_a).shift(1)   # own rating after the previous match
    t = t[(t[0] == 2026) & (t[7] == "WC")].copy()
    assert (t["elo_own"] == t["prev_post"]).all(), code
    t["n"] = range(1, len(t) + 1)
    t["code"] = code
    parts.append(t[["code", "n", "opp", "elo_own", "elo_opp"]])
elo = pd.concat(parts)

j = df.assign(ec=df["team"].map(elo_code), eo=df["opponent"].map(elo_code)).merge(
    elo, left_on=["ec", "team_match_no"], right_on=["code", "n"], how="left", validate="one_to_one")
assert (j["opp"] == j["eo"]).all(), "match order differs between FIFA and eloratings"

mirror = j.merge(j, left_on=["match_no", "opp_code"], right_on=["match_no", "team_code"])
assert (mirror["elo_own_x"] == mirror["elo_opp_y"]).all(), "team and opponent files disagree"

out = append_cols(j[["match_no", "team_code", "elo_own", "elo_opp"]])
print(out.shape)

(208, 21)


In [18]:
# Fifa keeps null entries, I am using manually collected data of venues and captial of each country
for f in ["raw/venue_altitudes_16.csv", "raw/capitals_48.csv"]:
    t = pd.read_csv(f)
    print(f, t.shape, t.columns.tolist())
    print(t.head(3).to_string(), "\n")

raw/venue_altitudes_16.csv (16, 4) ['venue_city', 'stadium_name', 'stadium_altitude_m', 'source_url']
  venue_city           stadium_name  stadium_altitude_m                                                                     source_url
0    Atlanta  Mercedes-Benz Stadium                 307  https://api.open-meteo.com/v1/elevation?latitude=33.75528&longitude=-84.40083
1     Boston       Gillette Stadium                  84      https://api.open-meteo.com/v1/elevation?latitude=42.091&longitude=-71.264
2     Dallas           AT&T Stadium                 193  https://api.open-meteo.com/v1/elevation?latitude=32.74769&longitude=-97.09288 

raw/capitals_48.csv (48, 5) ['fifa_code', 'team_name', 'capital', 'capital_altitude_m', 'source_url']
  fifa_code      team_name       capital  capital_altitude_m                                                                      source_url
0       USA  United States    Washington                  12   https://api.open-meteo.com/v1/elevation?latitude=3

In [19]:
from urllib.parse import urlparse, parse_qs

def coords(url):
    q = parse_qs(urlparse(url).query)
    return float(q["latitude"][0]), float(q["longitude"][0])

ELEV = Path("raw/elevation"); ELEV.mkdir(parents=True, exist_ok=True)
ven = pd.read_csv("raw/venue_altitudes_16.csv")
cap = pd.read_csv("raw/capitals_48.csv")
for t in (ven, cap):
    t["lat"], t["lon"] = zip(*t["source_url"].map(coords))

assert set(ven["venue_city"]) == set(df["city"]), set(ven["venue_city"]) ^ set(df["city"])
assert set(cap["fifa_code"]) == set(df["team_code"]), set(cap["fifa_code"]) ^ set(df["team_code"])

def elevation(t, name):
    el = fetch_json("https://api.open-meteo.com/v1/elevation",
                    {"latitude": ",".join(map(str, t["lat"])), "longitude": ",".join(map(str, t["lon"]))},
                    ELEV / f"{name}.json")
    return pd.Series(el["elevation"], index=t.index)

ven["alt_api"] = elevation(ven, "venues")
cap["alt_api"] = elevation(cap, "capitals")

for t, col in ((ven, "stadium_altitude_m"), (cap, "capital_altitude_m")):
    d = (t["alt_api"] - t[col]).abs()
    print(col, "max |diff| =", d.max(), "| rows > 1 m:", (d > 1).sum())

stadium_altitude_m max |diff| = 0.0 | rows > 1 m: 0
capital_altitude_m max |diff| = 0.0 | rows > 1 m: 0


In [20]:
# Altitude ascent
stad = ven.set_index("venue_city")["alt_api"]
home_alt = cap.set_index("fifa_code")["alt_api"]

venue_alt = df["city"].map(stad)
asc_own = (venue_alt - df["team_code"].map(home_alt)).clip(lower=0)
asc_opp = (venue_alt - df["opp_code"].map(home_alt)).clip(lower=0)

new = df[["match_no", "team_code"]].assign(alt_ascent_diff=(asc_own - asc_opp).round().astype(int))
out = append_cols(new)
print(out.shape, (out["alt_ascent_diff"].abs() > 500).sum(), out["alt_ascent_diff"].describe().round(0).to_dict())

(208, 22) 14 {'count': 208.0, 'mean': 0.0, 'std': 428.0, 'min': -2210.0, '25%': -5.0, '50%': 0.0, '75%': 5.0, 'max': 2210.0}


In [21]:
# Squad revision before wc2026, wikipedia, for squad average age
WIKI = Path("raw/wiki"); WIKI.mkdir(parents=True, exist_ok=True)
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_UA = {"User-Agent": "HIT140-student-project/1.0 (github.com/prabin466)"}
CUTOFF = "2026-06-10T23:59:00Z"

meta = fetch_json(WIKI_API, {"action": "query", "titles": "2026 FIFA World Cup squads",
                             "prop": "revisions", "rvprop": "ids|timestamp", "rvstart": CUTOFF,
                             "rvdir": "older", "rvlimit": 1, "format": "json", "formatversion": 2},
                  WIKI / "squads_revision_meta.json", WIKI_UA)
rev = meta["query"]["pages"][0]["revisions"][0]
assert pd.Timestamp(rev["timestamp"]) < first_kickoff

page = fetch_json(WIKI_API, {"action": "parse", "oldid": rev["revid"], "prop": "text|revid",
                             "format": "json", "formatversion": 2},
                  WIKI / f"squads_{rev['revid']}.json", WIKI_UA)
assert page["parse"]["revid"] == rev["revid"]
html = page["parse"]["text"]
print(rev, len(html))

{'revid': 1358786179, 'parentid': 1358785315, 'timestamp': '2026-06-10T23:30:23Z'} 2053411


In [22]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")
rows = []
for tr in soup.select("tr.nat-fs-player"):
    cells = tr.find_all(["td", "th"])
    bday = tr.select_one("span.bday")
    rows.append({"wiki_team": tr.find_previous("h3").get_text(strip=True),
                 "player": cells[2].get_text(" ", strip=True),
                 "dob": bday.get_text() if bday else None})
sq = pd.DataFrame(rows)

print(len(sq), sq["dob"].isna().sum())
print(sq["wiki_team"].value_counts().sort_index().to_string())
print(sq.head(5))

1246 0
wiki_team
Algeria                   26
Argentina                 25
Australia                 26
Austria                   25
Belgium                   26
Bosnia and Herzegovina    26
Brazil                    26
Canada                    26
Cape Verde                26
Colombia                  26
Croatia                   26
Curaçao                   26
Czech Republic            26
DR Congo                  26
Ecuador                   26
Egypt                     26
England                   26
France                    26
Germany                   26
Ghana                     26
Haiti                     26
Iran                      26
Iraq                      26
Ivory Coast               26
Japan                     26
Jordan                    26
Mexico                    26
Morocco                   26
Netherlands               26
New Zealand               26
Norway                    26
Panama                    26
Paraguay                  26
Portugal                  

In [23]:
# Map into our csv
WIKI_RENAME = {"Cape Verde": "Cabo Verde", "Czech Republic": "Czechia", "DR Congo": "Congo DR",
               "Iran": "IR Iran", "Ivory Coast": "Côte d'Ivoire", "South Korea": "Korea Republic",
               "Turkey": "Türkiye", "United States": "USA"}
sq["team"] = sq["wiki_team"].replace(WIKI_RENAME)
assert set(sq["team"]) == set(df["team"]), set(sq["team"]) ^ set(df["team"])
sq["dob"] = pd.to_datetime(sq["dob"])
ref_day = first_kickoff.tz_convert(None).normalize()     # opening day, derived from FIFA data

In [24]:
# The 10 Jun revision has two wrong DOBs (fixed in later revisions); correct them before matching on DOB.
fix = pd.read_csv("raw/dob_corrections.csv", parse_dates=["wikipedia_dob", "corrected_dob"])
hit = sq.reset_index().merge(fix, on=["player", "team"], validate="one_to_one")
assert len(hit) == len(fix), "correction for a player not in the squad list"
assert (hit["dob"] == hit["wikipedia_dob"]).all(), "Wikipedia DOB differs from the one being corrected"
sq.loc[hit["index"], "dob"] = hit["corrected_dob"].to_numpy()
sq["dob_fixed"] = sq.index.isin(hit["index"])
print(sq.loc[sq["dob_fixed"], ["team", "player", "dob"]].to_string())

        team           player        dob
155   Canada    Jayden Nelson 2002-09-26
959  Algeria  Yacine Titraoui 2003-07-26


Squad value

Source: dcaribou/transfermarkt-datasets (github.com/dcaribou/transfermarkt-datasets), a public dataset of Transfermarkt data.
`players` gives Transfermarkt IDs, names and dates of birth; `player_valuations` gives each player's dated valuation history.

In [25]:
def fetch_file(url, out):
    # Binary download
    if not out.exists():
        r = requests.get(url, headers=HEADERS, timeout=120)
        r.raise_for_status()
        out.write_bytes(r.content)
        row = pd.DataFrame([{"file": str(out), "url": r.url,
                             "fetched_utc": datetime.now(timezone.utc).isoformat(),
                             "sha256": hashlib.sha256(r.content).hexdigest()}])
        row.to_csv(MANIFEST, mode="a", header=not MANIFEST.exists(), index=False)
    return out

TM = Path("raw/transfermarkt"); TM.mkdir(parents=True, exist_ok=True)
DCB = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data"
pl = pd.read_csv(fetch_file(f"{DCB}/players.csv.gz", TM / "players.csv.gz"))
pv = pd.read_csv(fetch_file(f"{DCB}/player_valuations.csv.gz", TM / "player_valuations.csv.gz"))
print(pl.shape, pv.shape)
print(pv.columns.tolist())

(50149, 26) (656301, 6)
['player_id', 'date', 'market_value_in_eur', 'current_club_name', 'current_club_id', 'player_club_domestic_competition_id']


In [26]:
import unicodedata, re

def norm(s):
    # lower-case, strip accents and "(captain)"-style notes, split into words
    s = re.sub(r"\(.*?\)", " ", str(s))
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z ]", " ", s).split()

pl["dob"] = pd.to_datetime(pl["date_of_birth"], errors="coerce").dt.normalize()
pl["tok"] = pl["name"].map(norm)
by_dob = {d: g for d, g in pl.dropna(subset=["dob"]).groupby("dob")}

def match(player, dob):
    cand = by_dob.get(dob)
    if cand is None:
        return None, "no_dob"
    toks = norm(player)
    exact = cand[cand["tok"].map(lambda t: t == toks)]
    if len(exact) == 1:
        return exact["player_id"].iat[0], "T1"
    shared = cand[cand["tok"].map(lambda t: bool(set(t) & set(toks)))]
    if len(shared) == 1:
        return shared["player_id"].iat[0], "T2"
    return None, "ambiguous" if len(shared) > 1 else "no_name"

sq[["tm_id", "how"]] = [match(p, d) for p, d in zip(sq["player"], sq["dob"])]
assert not sq["tm_id"].dropna().duplicated().any(), "one Transfermarkt ID matched to two players"
print(sq["how"].value_counts().to_string())

how
T1           1135
T2             84
no_name        25
ambiguous       2


In [27]:
t2 = sq[sq["how"] == "T2"].merge(pl[["player_id", "name", "country_of_citizenship"]],
                                 left_on="tm_id", right_on="player_id")
odd = t2[t2["country_of_citizenship"].str.lower() != t2["wiki_team"].str.lower()]
print(len(t2), len(odd))
print(odd[["wiki_team", "player", "name", "country_of_citizenship"]].to_string())

84 26
      wiki_team                     player             name country_of_citizenship
1   South Korea              Kim Seung-gyu    Seung-gyu Kim           Korea, South
2   South Korea               Lee Han-beom     Han-beom Lee           Korea, South
3   South Korea                Lee Gi-hyuk      Gi-hyuk Lee           Korea, South
4   South Korea                Kim Min-jae      Min-jae Kim           Korea, South
5   South Korea              Kim Tae-hyeon    Tae-hyeon Kim           Korea, South
6   South Korea              Hwang In-beom    In-beom Hwang           Korea, South
7   South Korea  Son Heung-min ( captain )    Heung-min Son           Korea, South
8   South Korea              Paik Seung-ho    Seung-ho Paik           Korea, South
9   South Korea               Cho Gue-sung     Gue-sung Cho           Korea, South
10  South Korea               Lee Jae-sung     Jae-sung Lee           Korea, South
11  South Korea             Hwang Hee-chan   Hee-chan Hwang           Korea, Sout

In [28]:
CIT_ALIAS = {"korea, south": "south korea"}   # Transfermarkt's spelling of the same country
cit = (sq["tm_id"].map(pl.set_index("player_id")["country_of_citizenship"])
       .str.lower().replace(CIT_ALIAS))
reject = (sq["how"] == "T2") & (cit != sq["wiki_team"].str.lower())
sq.loc[reject, "tm_id"] = None
sq.loc[reject, "how"] = "T2_rejected"
print(sq.loc[reject, ["wiki_team", "player"]].to_string())
print(sq["how"].value_counts().to_string(), "\nmatched:", sq["tm_id"].notna().sum())

    wiki_team        player
675     Egypt  Mohamed Alaa
how
T1             1135
T2               83
no_name          25
ambiguous         2
T2_rejected       1 
matched: 1218


In [29]:
# Hand-checked IDs for players the name rules cannot reach (transliteration or ambiguous same-DOB names).
man = pd.read_csv("raw/manual_matches.csv")
hit = sq.reset_index().merge(man, on=["player", "team"], suffixes=("", "_man"), validate="one_to_one")
assert len(hit) == len(man), "manual match for a player not in the squad list"
clash = hit["tm_id"].notna() & (hit["tm_id"] != hit["tm_id_man"])
assert not clash.any(), hit.loc[clash, ["team", "player", "tm_id", "tm_id_man"]]
todo = hit[hit["tm_id"].isna()]
sq.loc[todo["index"], "tm_id"] = todo["tm_id_man"].to_numpy()
sq.loc[todo["index"], "how"] = "manual"
assert not sq["tm_id"].dropna().duplicated().any(), "manual ID already used by another player"
print(len(todo), "filled |", len(hit) - len(todo), "already auto-matched to the same ID")

5 filled | 0 already auto-matched to the same ID


In [30]:
sq["tm_id"] = sq["tm_id"].astype("Int64")
pv["date"] = pd.to_datetime(pv["date"])

pre = pv[pv["date"] < ref_day].sort_values("date").groupby("player_id").last()
sq["value_eur"] = sq["tm_id"].map(pre["market_value_in_eur"])
sq["value_date"] = sq["tm_id"].map(pre["date"])

no_val = sq["tm_id"].notna() & sq["value_eur"].isna()
print("matched but no pre-WC valuation:", no_val.sum())
print(sq["value_date"].describe())

matched but no pre-WC valuation: 60
count                             1163
mean     2026-04-25 00:39:37.300086016
min                2023-05-29 00:00:00
25%                2026-05-26 00:00:00
50%                2026-05-29 00:00:00
75%                2026-06-03 00:00:00
max                2026-06-10 00:00:00
Name: value_date, dtype: object


60 unmatched players, lets check

In [31]:
import numpy as np
TM_HDR = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)", "Accept": "application/json"}
API = TM / "api_mv"; API.mkdir(exist_ok=True)

def tm_value(pid):
    out = API / f"{pid}.json"
    cached = out.exists()
    j = fetch_json(f"https://www.transfermarkt.com/ceapi/marketValueDevelopment/graph/{pid}", None, out, TM_HDR)
    if not cached:
        time.sleep(2)          # polite pause only when we actually hit the site
    pts = pd.DataFrame(j.get("list", []))
    if pts.empty:
        return None, None
    pts["date"] = pd.to_datetime(pts["datum_mw"], format="%d/%m/%Y")
    pts = pts[pts["date"] < ref_day].sort_values("date")
    return (pts["y"].iat[-1], pts["date"].iat[-1]) if len(pts) else (None, None)

sq["value_src"] = np.where(sq["value_eur"].notna(), "dcaribou", None)
for i in sq.index[no_val]:
    v, d = tm_value(int(sq.at[i, "tm_id"]))
    sq.at[i, "value_eur"], sq.at[i, "value_date"] = v, d
    if v is not None:
        sq.at[i, "value_src"] = "tm_api"

print(sq["value_src"].value_counts(dropna=False).to_string())
stale = (ref_day - pd.to_datetime(sq["value_date"])).dt.days > 365
print("valued:", sq["value_eur"].notna().sum(), "| older than 1 year:", stale.sum())

value_src
dcaribou    1163
tm_api        60
None          23
valued: 1223 | older than 1 year: 41


Sample verification

In [32]:
# Same pool as the original check (auto-matched, uncorrected rows), so the sample stays fixed.
pool = sq[(sq["value_src"] == "dcaribou") & (sq["how"] != "manual") & ~sq["dob_fixed"]]
samp = pool.sample(50, random_state=140).copy()
api = [tm_value(int(pid)) for pid in samp["tm_id"]]
samp["api_value"] = [v for v, _ in api]
samp["api_date"] = [d for _, d in api]

agree = (samp["value_eur"] == samp["api_value"]) & (pd.to_datetime(samp["value_date"]) == samp["api_date"])
print(agree.sum(), "/ 50 agree")
print(samp.loc[~agree, ["team", "player", "value_eur", "api_value", "value_date", "api_date"]].to_string())

47 / 50 agree
        team                     player  value_eur  api_value value_date   api_date
678  IR Iran  Ehsan Hajsafi ( captain )   275000.0     250000 2025-12-08 2026-05-21
419  Curaçao           Roshon van Eijma   375000.0     375000 2025-12-16 2026-05-21
421  Curaçao        Godfried Roemeratoe   400000.0     350000 2025-12-16 2026-05-21


In [33]:
d = pd.to_datetime(samp["value_date"])
print(pd.crosstab(d < "2026-04-01", agree, rownames=["old (<Apr 2026)"], colnames=["agree"]))
print("players to refresh:", ((pd.to_datetime(sq["value_date"]) < "2026-04-01") & (sq["value_src"] == "dcaribou")).sum())

agree            False  True 
old (<Apr 2026)              
False                0     47
True                 3      0
players to refresh: 57


In [34]:
REFRESH_BEFORE = pd.Timestamp("2026-04-01")
old = (sq["value_src"] == "dcaribou") & (pd.to_datetime(sq["value_date"]) < REFRESH_BEFORE)

changed = 0
for i in sq.index[old]:
    v, d = tm_value(int(sq.at[i, "tm_id"]))
    if v is not None:
        changed += int(v != sq.at[i, "value_eur"] or d != pd.Timestamp(sq.at[i, "value_date"]))
        sq.at[i, "value_eur"], sq.at[i, "value_date"], sq.at[i, "value_src"] = v, d, "tm_api_refresh"

assert (pd.to_datetime(sq["value_date"].dropna()) < ref_day).all()
print("refreshed:", old.sum(), "| value or date changed:", changed)
print(sq["value_src"].value_counts(dropna=False).to_string())

refreshed:

 57 | value or date changed: 57
value_src
dcaribou          1106
tm_api              60
tm_api_refresh      57
None                23


In [35]:
# Team totals after the refresh; value and age both come from the corrected player table.
sq["value_eur"] = pd.to_numeric(sq["value_eur"])
sq["age"] = (ref_day - pd.to_datetime(sq["dob"])).dt.days / 365.25
stale = (ref_day - pd.to_datetime(sq["value_date"])).dt.days > 365
team_val = sq.groupby("team")["value_eur"].sum(min_count=1)
n_val = sq.groupby("team")["value_eur"].count()
age = sq.groupby("team")["age"].mean()

print("older than 1 year:", stale.sum(), "| share of value:",
      round(sq.loc[stale, "value_eur"].sum() / sq["value_eur"].sum(), 4))
print("fewest valued players in a team:", n_val.min(), n_val.idxmin())
cov = pd.DataFrame({"valued": n_val, "squad": sq.groupby("team").size()})
cov["coverage"] = (cov["valued"] / cov["squad"]).round(3)
print(cov.sort_values(["coverage", "valued"]).to_string())

older than 1 year: 0 | share of value: 0.0
fewest valued players in a team: 22 Jordan
                        valued  squad  coverage
team                                           
Jordan                      22     26     0.846
Saudi Arabia                22     26     0.846
IR Iran                     24     26     0.923
Tunisia                     24     26     0.923
Uzbekistan                  24     26     0.923
Algeria                     25     26     0.962
Bosnia and Herzegovina      25     26     0.962
Egypt                       25     26     0.962
Haiti                       25     26     0.962
Morocco                     25     26     0.962
Paraguay                    25     26     0.962
Qatar                       25     26     0.962
Scotland                    25     26     0.962
Senegal                     25     26     0.962
Argentina                   25     25     1.000
Austria                     25     25     1.000
Australia                   26     26     1.000
Be

In [36]:
# Squad features
base = pd.read_csv("obj2.2.csv")
new = base[["match_no", "team_code"]].assign(
    squad_value_own=base["team"].map(team_val), squad_value_opp=base["opponent"].map(team_val),
    squad_age_own=base["team"].map(age), squad_age_opp=base["opponent"].map(age))
out = append_cols(new)
saved = out.drop_duplicates("team").set_index("team")["squad_value_own"]
assert (saved - sq.groupby("team")["value_eur"].sum().reindex(saved.index)).abs().max() == 0, "csv value != player totals"
print(out.shape, round(age.min(), 2), round(age.max(), 2))

(208, 26) 25.85 30.43


In [37]:
# Home advantage
base = pd.read_csv("obj2.2.csv")
home_own = (base["team_code"] == base["venue_country"]).astype(int)
home_opp = (base["opp_code"] == base["venue_country"]).astype(int)

new = base[["match_no", "team_code"]].assign(home_own=home_own, home_opp=home_opp,
                                             home_venue=home_own - home_opp)
out = append_cols(new)
print(out.shape, home_own.sum(), base.loc[home_own == 1, "team_code"].value_counts().to_dict())

(208, 29) 13 {'MEX': 5, 'USA': 5, 'CAN': 3}


In [38]:
# Rest difference using timestamps of each team's last game, Match 1 for each team, it will be set to 0
b = pd.read_csv("obj2.2.csv", parse_dates=["kickoff_utc"]).sort_values(["team_code", "kickoff_utc"])
b["rest"] = b.groupby("team_code")["kickoff_utc"].diff().dt.total_seconds() / 86400

opp = b[["match_no", "team_code", "rest"]].rename(columns={"team_code": "opp_code", "rest": "opp_rest"})
b = b.merge(opp, on=["match_no", "opp_code"], validate="one_to_one")

first = b["rest"].isna() | b["opp_rest"].isna()
b["rest_diff"] = (b["rest"] - b["opp_rest"]).where(~first, 0.0).round(4)

assert b.groupby("match_no")["rest_diff"].sum().abs().max() < 1e-6, "own/opp rest misaligned"
out = append_cols(b[["match_no", "team_code", "rest_diff"]])
print(out.shape, b["rest_diff"].abs().max(), first.sum())

(208, 30) 3.2917 48


Jet-lag feature from each venue rather than intercontinet as team arrive weeks before , jet-lag has been adjusted. Base-camps were listed online and collected manually.

In [39]:
bc = pd.read_csv("raw/base_camps_48.csv")
print(bc.shape, bc.columns.tolist())
print(bc.head(3).to_string())
print(sorted(pd.read_csv("obj2.2.csv")["city"].unique()))

(48, 5) ['fifa_code', 'team_name', 'base_camp_city', 'base_camp_utc_offset_jun2026', 'source_url']
  fifa_code      team_name   base_camp_city  base_camp_utc_offset_jun2026                                                                                                  source_url
0       USA  United States       Irvine, CA                            -7  https://www.ussoccer.com/stories/2026/06/usmnt/where-are-fifa-world-cup-2026-team-base-camp-training-sites
1       ALG        Algeria     Lawrence, KS                            -5  https://www.ussoccer.com/stories/2026/06/usmnt/where-are-fifa-world-cup-2026-team-base-camp-training-sites
2       ARG      Argentina  Kansas City, KS                            -5  https://www.ussoccer.com/stories/2026/06/usmnt/where-are-fifa-world-cup-2026-team-base-camp-training-sites
['Atlanta', 'Boston', 'Dallas', 'Guadalajara', 'Houston', 'Kansas City', 'Los Angeles', 'Mexico City', 'Miami', 'Monterrey', 'New Jersey', 'Philadelphia', 'San Francisco Bay

In [40]:
from zoneinfo import ZoneInfo

CITY_TZ = {"Atlanta": "America/New_York", "Boston": "America/New_York", "Dallas": "America/Chicago",
           "Guadalajara": "America/Mexico_City", "Houston": "America/Chicago", "Kansas City": "America/Chicago",
           "Los Angeles": "America/Los_Angeles", "Mexico City": "America/Mexico_City", "Miami": "America/New_York",
           "Monterrey": "America/Monterrey", "New Jersey": "America/New_York", "Philadelphia": "America/New_York",
           "San Francisco Bay Area": "America/Los_Angeles", "Seattle": "America/Los_Angeles",
           "Toronto": "America/Toronto", "Vancouver": "America/Vancouver"}

b = pd.read_csv("obj2.2.csv", parse_dates=["kickoff_utc"]).sort_values(["team_code", "kickoff_utc"])
assert set(b["city"]) == set(CITY_TZ)
b["voff"] = [k.tz_convert(ZoneInfo(CITY_TZ[c])).utcoffset().total_seconds() / 3600
             for k, c in zip(b["kickoff_utc"], b["city"])]

bc = pd.read_csv("raw/base_camps_48.csv")
assert set(bc["fifa_code"]) == set(b["team_code"])
assert set(bc["base_camp_utc_offset_jun2026"]) <= {-4, -5, -6, -7}, "unexpected base-camp offset"
b["bcoff"] = b["team_code"].map(bc.set_index("fifa_code")["base_camp_utc_offset_jun2026"])

b["prev_voff"] = b.groupby("team_code")["voff"].shift(1)
b["shift"] = (b["voff"] - b["bcoff"]).abs().where(b["stage"] == "Group", (b["voff"] - b["prev_voff"]).abs())

opp = b[["match_no", "team_code", "shift"]].rename(columns={"team_code": "opp_code", "shift": "opp_shift"})
b = b.merge(opp, on=["match_no", "opp_code"], validate="one_to_one")
b["tz_shift_diff"] = (b["shift"] - b["opp_shift"]).astype(int)

assert b.groupby("match_no")["tz_shift_diff"].sum().abs().max() == 0
out = append_cols(b[["match_no", "team_code", "tz_shift_diff"]])
print(out.shape, b["voff"].value_counts().to_dict(), b["tz_shift_diff"].value_counts().sort_index().to_dict())

(208, 31) {-4.0: 84, -7.0: 54, -5.0: 44, -6.0: 26} {-3: 2, -2: 12, -1: 33, 0: 114, 1: 33, 2: 12, 3: 2}


In [41]:
# Pre wc priors, to fill in data for match 1, we went looking for last 10 competitive games, so that we could mathc the stakes of wc, data is verfied against FIFA and wikipedia.
l10 = pd.read_csv("raw/wc2026_last10_competitive.csv", parse_dates=["date"])
base = pd.read_csv("obj2.2.csv")

assert len(l10) == 480 and (l10.groupby("team").size() == 10).all()
assert set(l10["team"]) == set(base["team"]), set(l10["team"]) ^ set(base["team"])
assert (l10["date"] < ref_day).all(), "a prior match is not before the tournament"

pri = l10.groupby("team").agg(gf=("goals_for_90", "mean"), ga=("goals_against_90", "mean"))
new = base[["match_no", "team_code"]].assign(
    prior_gf=base["team"].map(pri["gf"]), prior_ga=base["team"].map(pri["ga"]),
    opp_prior_gf=base["opponent"].map(pri["gf"]), opp_prior_ga=base["opponent"].map(pri["ga"]))
out = append_cols(new)
print(out.shape, l10["date"].min().date(), l10["date"].max().date())

(208, 35) 2022-06-14 2026-03-31


In [42]:
# XG was also collected manually from The stats dont Lie, compared against FoTMob.
cor_t = pd.read_csv("raw/tsdl_xg_corrected.csv")

TSDL_RENAME = {"South Korea": "Korea Republic", "Czech Republic": "Czechia", "Cape Verde": "Cabo Verde",
               "Ivory Coast": "Côte d'Ivoire", "Iran": "IR Iran", "Turkey": "Türkiye", "United States": "USA",
               "Bosnia & Herzegovina": "Bosnia and Herzegovina", "Curacao": "Curaçao", "D.R. Congo": "Congo DR"}
t = cor_t.replace({"home_team": TSDL_RENAME, "away_team": TSDL_RENAME})
t["tsdl_date"] = pd.to_datetime(t["date"], format="%d/%m/%Y")

base = pd.read_csv("obj2.2.csv", parse_dates=["kickoff_utc"])
assert set(t["home_team"]) | set(t["away_team"]) <= set(base["team"]), \
    (set(t["home_team"]) | set(t["away_team"])) - set(base["team"])

long = pd.concat([
    t.rename(columns={"home_team": "team", "away_team": "opponent", "home_xg": "xg_for", "away_xg": "xg_against"}),
    t.rename(columns={"away_team": "team", "home_team": "opponent", "away_xg": "xg_for", "home_xg": "xg_against"})])
m = base.merge(long[["team", "opponent", "tsdl_date", "xg_for", "xg_against"]], on=["team", "opponent"], how="left")
m = m[m["tsdl_date"].isna() | ((m["tsdl_date"] - m["kickoff_utc"].dt.tz_convert(None).dt.normalize()).dt.days.abs() <= 2)]
assert m.duplicated(["match_no", "team_code"]).sum() == 0 and len(m) == 208
print("rows without xG:", m["xg_for"].isna().sum(), m.loc[m["xg_for"].isna(), ["stage", "team"]].to_dict("records"))

rows without xG: 2 [{'stage': '3rd place', 'team': 'England'}, {'stage': '3rd place', 'team': 'France'}]


The third-place match is absent from the xG source; as it is both teams’ final match, no lagged feature depends on it.

In [43]:
# Lagged in-tournament Xg
m = m.sort_values(["team_code", "kickoff_utc"])
m["min90"] = np.where(m["went_to_et"], 120 / 90, 1.0)

def lagged(col):
    # cumulative total up to and including each match, shifted down one so a row only sees earlier matches
    return m.groupby("team_code")[col].cumsum().groupby(m["team_code"]).shift(1).fillna(0)

m["sum_xg_before"] = lagged("xg_for")
m["sum_xga_before"] = lagged("xg_against")
m["min90_before"] = lagged("min90")
m["n_before"] = m["team_match_no"] - 1

own = ["n_before", "min90_before", "sum_xg_before", "sum_xga_before"]
opp = m[["match_no", "team_code"] + own].rename(columns={"team_code": "opp_code", **{c: f"opp_{c}" for c in own}})
m = m.merge(opp, on=["match_no", "opp_code"], validate="one_to_one")

assert (m.loc[m["team_match_no"] == 1, own] == 0).all().all(), "first match must have no history"
out = append_cols(m[["match_no", "team_code"] + own + [f"opp_{c}" for c in own]])
print(out.shape, m["n_before"].max(), round(m["min90_before"].max(), 2))

(208, 43)

 7 7.67


In [44]:
# Validation: final checks on obj2.2.csv; every assert must pass before the file is used downstream.
v = pd.read_csv("obj2.2.csv")

assert len(v) == 208, len(v)
assert v["match_no"].nunique() == 104
assert (v.groupby("match_no").size() == 2).all()

# Row pairs within a match: opponent's row is the mirror image of this one.
pairs = [(c, c[:-4] + "_opp") for c in v.columns if c.endswith("_own")]
pairs += [(c, "opp_" + c) for c in v.columns if "opp_" + c in v.columns]
assert all(o in v.columns for _, o in pairs)
mir = v.merge(v, left_on=["match_no", "opp_code"], right_on=["match_no", "team_code"],
              suffixes=("", "_m"), validate="one_to_one")
assert len(mir) == 208 and (mir["opponent"] == mir["team_m"]).all()
for own, opp in pairs:
    assert (mir[own] == mir[opp + "_m"]).all(), (own, opp)
diffs = [c for c in v.columns if c.endswith("_diff")] + ["home_venue"]   # home_venue is own − opp too
dsum = v.groupby("match_no")[diffs].sum().abs().max()
assert (dsum <= 1e-9).all(), dsum[dsum > 1e-9]

per_match = v.drop_duplicates("match_no")["stage"].value_counts().to_dict()
assert per_match == {"Group": 72, "Round of 32": 16, "Round of 16": 8, "QF": 4, "SF": 2,
                     "3rd place": 1, "Final": 1}, per_match

assert (v["goals_90"] <= v["goals"]).all()
assert not ((v["goals_90"] != v["goals"]) & ~v["went_to_et"]).any(), "goals_90 changed without extra time"

assert v.notna().all().all(), v.columns[v.isna().any()].tolist()
print(v.dtypes.to_string())
assert (v[["squad_value_own", "squad_value_opp"]] > 0).all().all()
# Raw altitudes are not stored in the csv, so check the source tables and the derived diff bound.
assert ven["alt_api"].between(0, 3000).all() and cap["alt_api"].between(0, 3000).all()
assert v["alt_ascent_diff"].abs().max() <= 3000
for c in ("goals", "goals_90"):
    assert pd.api.types.is_integer_dtype(v[c]) and (v[c] >= 0).all(), c

alts = pd.concat([ven["alt_api"], cap["alt_api"]])
print(f"\nAll checks passed: {len(v)} rows, {len(pairs)} mirror pairs, {len(diffs)} diff columns, "
      f"stages {per_match}, altitudes {alts.min():.0f}–{alts.max():.0f} m")

match_no                int64
fifa_match_id           int64
kickoff_utc            object
stage                  object
team                   object
team_code              object
opponent               object
opp_code               object
team_match_no           int64
goals                   int64
goals_90                int64
went_to_et               bool
knockout                int64
stage_ord               int64
venue                  object
city                   object
venue_country          object
fifa_points_own       float64
fifa_points_opp       float64
elo_own                 int64
elo_opp                 int64
alt_ascent_diff         int64
squad_value_own       float64
squad_value_opp       float64
squad_age_own         float64
squad_age_opp         float64
home_own                int64
home_opp                int64
home_venue              int64
rest_diff             float64
tz_shift_diff           int64
prior_gf              float64
prior_ga              float64
opp_prior_


All checks passed: 208 rows, 11 mirror pairs, 4 diff columns, stages {'Group': 72, 'Round of 32': 16, 'Round of 16': 8, 'QF': 4, 'SF': 2, '3rd place': 1, 'Final': 1}, altitudes 1–2858 m
